<a href="https://colab.research.google.com/github/gouthamsankar-clg/BIS_Lab/blob/main/Lab_4/CuckooSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import math

def portfolio_objective(w, mu, Sigma, lam):
    portfolio_variance = w.T @ Sigma @ w
    expected_return = mu @ w
    return portfolio_variance - (lam * expected_return)

def enforce_constraints(w, lo, hi):
    # FIXED: Strict iterative allocation to ensure upper bounds are never breached
    n = len(w)
    w_fixed = np.clip(w, lo, hi)

    # Loop to properly distribute overflow cash without breaching 'hi'
    for _ in range(n):
        current_sum = np.sum(w_fixed)
        if np.isclose(current_sum, 1.0):
            break

        # Calculate how much cash is missing or extra
        diff = 1.0 - current_sum
        # Find which assets are not locked at the boundary ceilings/floors
        adjustable = (w_fixed > lo) & (w_fixed < hi)

        if not np.any(adjustable):
            # If all are locked at bounds, force equal weight of remaining space
            w_fixed = np.clip(w_fixed + diff / n, lo, hi)
            break

        w_fixed[adjustable] += diff / np.sum(adjustable)
        w_fixed = np.clip(w_fixed, lo, hi)

    return w_fixed

def generate_levy_step(beta=1.5):
    num = math.gamma(1 + beta) * np.sin(np.pi * beta / 2)
    den = math.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2))
    sigma_u = (num / den) ** (1 / beta)
    u = np.random.normal(0, sigma_u)
    v = np.random.normal(0, 1)
    return u / (np.abs(v) ** (1 / beta))

def run_cuckoo_search(returns_matrix, lam, lo, hi, budget, n_nests=25, pa=0.25, alpha=0.01, max_iter=200):
    mu = np.mean(returns_matrix, axis=0)
    Sigma = np.cov(returns_matrix, rowvar=False)
    n_assets = returns_matrix.shape[1]

    nests = np.array([enforce_constraints(np.random.rand(n_assets), lo, hi) for _ in range(n_nests)])
    scores = np.array([portfolio_objective(nest, mu, Sigma, lam) for nest in nests])

    best_idx = np.argmin(scores)
    global_best_pos = nests[best_idx].copy()
    global_best_score = scores[best_idx]

    for t in range(max_iter):
        for i in range(n_nests):
            step = generate_levy_step()
            new_pos = nests[i] + alpha * step * (nests[i] - global_best_pos)
            new_pos = enforce_constraints(new_pos, lo, hi)

            j = np.random.randint(0, n_nests)
            new_score = portfolio_objective(new_pos, mu, Sigma, lam)
            if new_score < scores[j]:
                nests[j] = new_pos
                scores[j] = new_score

        worst_nests_sorted = np.argsort(scores)[::-1]
        num_to_abandon = int(np.ceil(pa * n_nests))
        for k in range(num_to_abandon):
            target_idx = worst_nests_sorted[k]
            nests[target_idx] = enforce_constraints(np.random.rand(n_assets), lo, hi)
            scores[target_idx] = portfolio_objective(nests[target_idx], mu, Sigma, lam)

        current_best_idx = np.argmin(scores)
        if scores[current_best_idx] < global_best_score:
            global_best_score = scores[current_best_idx]
            global_best_pos = nests[current_best_idx].copy()

    w_star = enforce_constraints(global_best_pos, lo, hi)
    return w_star, budget * w_star, mu @ w_star, np.sqrt(w_star.T @ Sigma @ w_star)

# --- REPRODUCIBLE DATA GENERATION ---
np.random.seed(42)
days = 252
sample_returns = np.hstack([
    np.random.normal(0.0015, 0.020, (days, 1)),
    np.random.normal(0.0002, 0.005, (days, 1)),
    np.random.normal(0.0008, 0.012, (days, 1)),
    np.random.normal(0.0004, 0.014, (days, 1))
])

print(">>> CUCKOO SEARCH INPUT DECK SETUP <<<")
TOTAL_BUDGET = float(input("Enter Total Budget Cash Pool: "))
LAMBDA_PENALTY = float(input("Enter Risk Penalty Lambda Dial: "))
MIN_BOUND = float(input("Enter Minimum Asset Bound Floor: "))
MAX_BOUND = float(input("Enter Maximum Asset Bound Cap: "))

cs_w, cs_cash, cs_ret, cs_risk = run_cuckoo_search(sample_returns, LAMBDA_PENALTY, MIN_BOUND, MAX_BOUND, TOTAL_BUDGET)

labels = ['Asset 0', 'Asset 1', 'Asset 2', 'Asset 3']
output_df = pd.DataFrame({'Cuckoo Weight (%)': np.round(cs_w * 100, 2), 'Cuckoo Cash ($)': np.round(cs_cash, 2)}, index=labels)
print("\n" + "="*50 + "\n             CUCKOO SEARCH ALGORITHM RESULTS\n" + "="*50)
print(output_df.to_string())
print(f"-" * 50 + f"\nAnnual Return: {cs_ret*days*100:.2f}%\nAnnual Risk Volatility: {cs_risk*np.sqrt(days)*100:.2f}%\n" + "="*50)


>>> CUCKOO SEARCH INPUT DECK SETUP <<<
Enter Total Budget Cash Pool: 100000
Enter Risk Penalty Lambda Dial: 0.12
Enter Minimum Asset Bound Floor: 0.0
Enter Maximum Asset Bound Cap: 0.85

             CUCKOO SEARCH ALGORITHM RESULTS
         Cuckoo Weight (%)  Cuckoo Cash ($)
Asset 0               15.0          15000.0
Asset 1                0.0              0.0
Asset 2                0.0              0.0
Asset 3               85.0          85000.0
--------------------------------------------------
Annual Return: 58.72%
Annual Risk Volatility: 18.31%
